In [1]:
import urllib.request
import ssl
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Create an unverified SSL context
    ssl_context = ssl._create_unverified_context()

    # Downloading the file
    with urllib.request.urlopen(url, context=ssl_context) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Unzipping the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add .tsv file extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


File downloaded and saved as sms_spam_collection/SMSSpamCollection.tsv


In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv(data_file_path,sep="\t",header=None,names=["Labels","Text"])
df["Labels"]=np.where(df["Labels"]=="spam",1,0)


In [3]:
df.head()

,Labels,Text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
ham=df[df["Labels"]==0].sample(n=df["Labels"].value_counts()[1])
data=pd.concat([ham,df[df["Labels"]==1]],ignore_index=True)

In [5]:
def split(data,size):

    train_size=int(data.shape[0]*size)
    test_size=data.shape[0]-train_size

    train=data.sample(n=train_size) # Removed ignore_index=True
    remaining_data = data.drop(train.index)
    valid=remaining_data.sample(n=int(test_size/2)) # Removed ignore_index=True
    test=remaining_data.drop(valid.index)

    # Reset indices after all splits are done
    train = train.reset_index(drop=True)
    valid = valid.reset_index(drop=True)
    test = test.reset_index(drop=True)

    return train,valid,test

In [6]:
train,valid,test=split(data,0.7)

In [7]:
import torch as th
from torch.utils.data import Dataset

class SpamData(Dataset):
    def __init__(self,dataframe,tokenizer,max_length=None,pad_token_id=50256):
        super().__init__()

        self.data=dataframe
        self.max_length=max_length
        self.encoded_tokens=[
            tokenizer.encode(text) for text in self.data["Text"]
        ]
        self.encoded_tokens_max=[
            token[:self.max_length] for token in self.encoded_tokens
        ]

        self.encoded_input_token=[

                 encoded_text+[pad_token_id]*(self.max_length-len(encoded_text))
                        for encoded_text in self.encoded_tokens_max
        ]

    def __getitem__(self,index):

        encoded=self.encoded_input_token[index]
        label=self.data.iloc[index]["Labels"]

        return (
            th.tensor(encoded,dtype=th.long),
            th.tensor(label,dtype=th.long)
        )
    def __len__(self):
        return self.data.shape[0]




In [8]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")

In [9]:
train_data=SpamData(train,tokenizer,max_length=100)
valid_data=SpamData(valid,tokenizer,max_length=100)
test_data=SpamData(test,tokenizer,max_length=100)

In [10]:
from torch.utils.data import DataLoader

num_workers=0
batch_size=10

th.manual_seed(416)
train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=True,num_workers=num_workers)
valid_loader=DataLoader(valid_data,batch_size=batch_size,shuffle=True,num_workers=num_workers)
test_loader=DataLoader(test_data,batch_size=batch_size,shuffle=True,num_workers=num_workers)


In [11]:
next(iter(train_loader))[0].shape

torch.Size([10, 100])

In [12]:
from transformers import GPT2LMHeadModel

# 1. Download the official OpenAI GPT-2 124M weights
hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_model.eval()

# 2. Extract the state dictionary containing all weight tensors
params = hf_model.state_dict()

# 3. View the available parameter names (keys)
for key in list(params.keys())[:10]:
    print(f"Key: {key:<40} Shape: {params[key].shape}")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Key: transformer.wte.weight                   Shape: torch.Size([50257, 768])
Key: transformer.wpe.weight                   Shape: torch.Size([1024, 768])
Key: transformer.h.0.ln_1.weight              Shape: torch.Size([768])
Key: transformer.h.0.ln_1.bias                Shape: torch.Size([768])
Key: transformer.h.0.attn.c_attn.weight       Shape: torch.Size([768, 2304])
Key: transformer.h.0.attn.c_attn.bias         Shape: torch.Size([2304])
Key: transformer.h.0.attn.c_proj.weight       Shape: torch.Size([768, 768])
Key: transformer.h.0.attn.c_proj.bias         Shape: torch.Size([768])
Key: transformer.h.0.ln_2.weight              Shape: torch.Size([768])
Key: transformer.h.0.ln_2.bias                Shape: torch.Size([768])


In [13]:
def generate_text(gpt,idx,context_size,new_max_tokens):


    for _ in range(new_max_tokens):
        # Ensure the input to the model does not exceed context_size
        # Take the last `context_size` tokens, or fewer if the sequence is shorter.
        input_to_gpt = idx[:, - context_size:]

        with th.no_grad():
            # Pass the context-limited input to the model
            logits=gpt(input_to_gpt)

        # Get the logits for the last token in the processed sequence
        logits=logits[:,-1,:]

        top_k_logits,top_k_tokens=th.topk(logits,1)

        min_prob=top_k_logits.min()
        logits=th.where(condition=logits<min_prob,other=logits,input=th.tensor(-th.inf).to(logits.device))

        scaled_logits=logits/1.3
        probs=th.nn.functional.softmax(scaled_logits,dim=-1)
        nxt_idx=th.multinomial(probs,num_samples=1)


        idx=th.cat((idx,nxt_idx),dim=1)

    return idx

In [14]:
def load_weights(model, params):

    # Token and positional embeddings
    model.token_embed.weight.data = params["transformer.wte.weight"]
    model.positional_embed.weight.data = params["transformer.wpe.weight"]

    # Transformer blocks
    for b_idx, block in enumerate(model.transformer_block):

        prefix = f"transformer.h.{b_idx}"

        # -------------------------
        # LayerNorm 1
        # -------------------------
        block.norm1.scale.data = params[f"{prefix}.ln_1.weight"]
        block.norm1.shift.data = params[f"{prefix}.ln_1.bias"]

        # -------------------------
        # QKV
        # -------------------------
        qkv_weight = params[f"{prefix}.attn.c_attn.weight"]
        qkv_bias = params[f"{prefix}.attn.c_attn.bias"]

        q_w, k_w, v_w = qkv_weight.split(768, dim=-1)
        q_b, k_b, v_b = qkv_bias.split(768, dim=0)

        block.mha.wq.weight.data = q_w.T
        block.mha.wk.weight.data = k_w.T
        block.mha.wv.weight.data = v_w.T

        block.mha.wq.bias.data = q_b
        block.mha.wk.bias.data = k_b
        block.mha.wv.bias.data = v_b

        # -------------------------
        # Attention output projection
        # -------------------------
        block.mha.proj.weight.data = (
            params[f"{prefix}.attn.c_proj.weight"].T
        )

        block.mha.proj.bias.data = (
            params[f"{prefix}.attn.c_proj.bias"]
        )

        # -------------------------
        # LayerNorm 2
        # -------------------------
        block.norm2.scale.data = params[f"{prefix}.ln_2.weight"]
        block.norm2.shift.data = params[f"{prefix}.ln_2.bias"]

        # -------------------------
        # Feed Forward
        # -------------------------
        block.ffnn.layers[0].weight.data = (
            params[f"{prefix}.mlp.c_fc.weight"].T
        )

        block.ffnn.layers[0].bias.data = (
            params[f"{prefix}.mlp.c_fc.bias"]
        )

        block.ffnn.layers[2].weight.data = (
            params[f"{prefix}.mlp.c_proj.weight"].T
        )

        block.ffnn.layers[2].bias.data = (
            params[f"{prefix}.mlp.c_proj.bias"]
        )

    # Final LayerNorm
    model.layer_norm.scale.data = params["transformer.ln_f.weight"]
    model.layer_norm.shift.data = params["transformer.ln_f.bias"]

    # GPT-2 ties output weights to token embeddings
    model.output.weight.data = model.token_embed.weight.data

    model.eval()

In [15]:
from torch import nn

class GPTModel(th.nn.Module):
    def __init__ (self,config):
        super().__init__()

        self.config=config

        self.token_embed=th.nn.Embedding(config["vocab_size"],config["embed_dim"])
        self.positional_embed=th.nn.Embedding(config["context_length"],config["embed_dim"])
        self.dropout=th.nn.Dropout(config["dropout_rate"])

        self.transformer_block=th.nn.Sequential(*[Transformer(config) for _ in range(config["num_layers"])])
        self.layer_norm=LayerNorm(config["embed_dim"])
        self.output=th.nn.Linear(config["embed_dim"],config["vocab_size"],bias=False)

    def forward (self,x):
        # x is expected to be (batch_size, sequence_length) containing token IDs
        token_embed=self.token_embed(x)
        positional_embed=self.positional_embed(th.arange(x.shape[1],device=x.device))
        x=token_embed+positional_embed
        x=self.dropout(x)
        x=self.transformer_block(x)
        x=self.layer_norm(x)
        logits=self.output(x)
        return logits

class MultiHeadAttention(th.nn.Module):
    def __init__(self,d_in,d_out,dropout,num_heads,context_length,qkv_bias=False):
        super().__init__()
        assert (d_out% num_heads==0), "d_out must be divisible by num_heads"
        self.dropout=nn.Dropout(dropout)
        self.d_out=d_out
        self.wk=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.wq=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.wv=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.num_heads=num_heads
        self.proj=nn.Linear(d_out,d_out)

        self.register_buffer("mask",th.triu(th.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b,num_tokens,d_in=x.shape

        keys=self.wk(x)
        queries=self.wq(x)
        values=self.wv(x)

        keys=keys.view(b,num_tokens,self.num_heads,self.d_out//self.num_heads)
        queries=queries.view(b,num_tokens,self.num_heads,self.d_out//self.num_heads)
        values=values.view(b,num_tokens,self.num_heads,self.d_out//self.num_heads)

        keys=keys.transpose(-3,-2)
        queries=queries.transpose(-3,-2)
        values=values.transpose(-3,-2)

        attention_scores=queries @ keys.transpose(2,3)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-th.inf)

        attention_weights=th.softmax(attention_scores/keys.shape[-1]**0.5,dim=-1)

        norm=self.dropout(attention_weights)

        context_vector =(norm @ values).transpose(1,2)

        context_vector=context_vector.contiguous().view(b,num_tokens,self.d_out)

        context_vector = self.proj(context_vector)


        context_vector_proj=self.dropout(context_vector)

        return context_vector_proj


class Transformer(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.mha=MultiHeadAttention(
            d_in=config["embed_dim"],
            d_out=config["embed_dim"],
            num_heads=config["num_heads"],
            dropout=config["dropout_rate"],
            context_length=config["context_length"],
            qkv_bias=config["qkv_bias"]
        )
        self.ffnn=FeedForwardLayer(config)
        self.norm1=LayerNorm(embed_dim=config["embed_dim"],epsilon=1e-5)
        self.norm2=LayerNorm(embed_dim=config["embed_dim"],epsilon=1e-5)
        self.dropout_rate=nn.Dropout(config["dropout_rate"])

    def forward(self,x):

        input=x

        x=self.norm1(x)
        x=self.mha(x)
        x=self.dropout_rate(x)
        x=x+input



        input=x

        x=self.norm2(x)
        x=self.ffnn(x)
        x=self.dropout_rate(x)
        x=x+input

        return x

class GeLU(th.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (
            1 + th.tanh(
                (2 / th.pi) ** 0.5 *
                (x + 0.044715 * x**3)
            )
        )

class FeedForwardLayer(th.nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config # Store config as an instance variable
        self.layers=th.nn.Sequential(
                th.nn.Linear(self.config["embed_dim"],4*self.config["embed_dim"]),
                GeLU(),
                th.nn.Linear(4*self.config["embed_dim"],self.config["embed_dim"])
        )

    def forward(self,x):
        return self.layers(x)



class LayerNorm(th.nn.Module):
    def __init__ (self,embed_dim,epsilon=1e-5):
        super().__init__()
        self.epsilon=epsilon
        self.scale=th.nn.Parameter(th.ones(embed_dim))
        self.shift=th.nn.Parameter(th.zeros(embed_dim))

    def forward (self,x):
        x_mean=x.mean(dim=-1,keepdim=True)
        x_variance=x.var(dim=-1,keepdim=True,unbiased=False)
        x_norm=(x-x_mean)/th.sqrt(x_variance+self.epsilon)

        return self.scale*x_norm + self.shift

In [16]:
GPT_CONFIG_124M = {
    "vocab_size":50257,
    "context_length":100,
    "embed_dim":768,
    "num_heads":12,
    "num_layers":12,
    "dropout_rate":0.1,
    "qkv_bias":True
}

In [17]:
model=GPTModel(GPT_CONFIG_124M)
load_weights(model,params)

In [18]:

import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [19]:
input="""
    classify following text 'spam' or 'ham' :
    Text : 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award - The Text is classified as   """

In [20]:
input_ids = tokenizer.encode(input)

print(input_ids)
decoded_ids=generate_text(model,th.tensor(input_ids).unsqueeze(0),context_size=100,new_max_tokens=16)

[198, 220, 220, 220, 36509, 1708, 2420, 705, 2777, 321, 6, 393, 705, 2763, 6, 1058, 198, 220, 220, 220, 8255, 1058, 705, 1639, 389, 257, 8464, 345, 423, 587, 20905, 6163, 284, 3328, 720, 12825, 5003, 393, 257, 720, 11024, 5764, 532, 383, 8255, 318, 10090, 355, 220, 220, 220]


In [21]:
tokenizer.decode(decoded_ids[0].tolist())

"\n    classify following text 'spam' or 'ham' :\n    Text : 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award - The Text is classified as    spam 'spam' or 'ham'\n\n'spam' or"

1.  Final OutputHead
2. Final TransformerBlock
2. Final LayerNorm Module

In [22]:
for param in model.parameters():
    param.requires_grad=False

In [23]:
th.manual_seed(416)

num_classes=2
model.output=th.nn.Linear(in_features=GPT_CONFIG_124M["embed_dim"],out_features=num_classes)

In [24]:
for param in model.transformer_block[-1].parameters():
    param.requires_grad=True

for param in model.layer_norm.parameters():
    param.requires_grad=True

In [25]:
with th.no_grad():
    output=model(th.tensor(input_ids).unsqueeze(0))
output.shape

torch.Size([1, 51, 2])

In [26]:
probs=th.softmax(output[:,-1,:],dim=-1)
print(probs)
prediction=th.argmax(probs,dim=-1)
print(prediction.item())

tensor([[0.0987, 0.9013]])
1


In [27]:
def Calculate_Accuracy(model,dataloader,num_batches=4):
    model.eval()
    correct_predictions,num_samples,wrong_predictions=0,0,0

    # num_batches=len(dataloader)

    for i, (input_batch,output_batch) in enumerate(dataloader):

        if i < num_batches:

            with th.no_grad():
                output=model(input_batch)
            probs=th.softmax(output[:,-1,:],dim=-1)
            prediction=th.argmax(probs,dim=-1)

            num_samples+=prediction.shape[0]
            correct_predictions+=(prediction==output_batch).sum().item()
            wrong_predictions+=(prediction!=output_batch).sum().item()


        else:
            break

    accuracy=correct_predictions/num_samples
    return accuracy,correct_predictions,wrong_predictions


In [28]:
def Cross_Entropy_Loss(model,input_batch,output_batch):

    logits=model(input_batch)
    loss=th.nn.functional.cross_entropy(logits[:,-1,:],output_batch)

    return loss


In [29]:
def Cross_Entropy(model,dataloader,num_batch=4):

    # model.eval()
    total_loss=0

    num_batches = num_batch if num_batch  is None else len(dataloader)

    for i, (input_batch,output_batch) in enumerate(dataloader):

        if i < num_batches:

            with th.no_grad():
                loss=Cross_Entropy_Loss(model,input_batch,output_batch)
            total_loss+=loss.item()

        else:
            break

    # model.train()

    return total_loss/num_batches



In [30]:
with th.no_grad():
    train_loss=Cross_Entropy(model,train_loader)
    valid_loss=Cross_Entropy(model,valid_loader)
    test_loss=Cross_Entropy(model,test_loader)

print("Training Loss :",train_loss)
print("Validation Loss :",valid_loss)
print("Test Loss :",test_loss)

Training Loss : 1.796697575705392
Validation Loss : 1.9051987813866658
Test Loss : 1.8757604943669361


In [31]:
def evaluate(model,train_loader,vali_loader):
    model.eval()
    with th.no_grad():
        train_loss=Cross_Entropy(model,train_loader)
        valid_loss=Cross_Entropy(model,vali_loader)
    model.train()
    return train_loss,valid_loss

In [32]:
def Training_Loop(model,train_loader,val_loader,optimizer,num_epochs=5,eval_freq=5):

    train_losses,valid_losses,train_acc,valid_acc=[],[],[],[]
    examples_seen,global_step=0,-1

    for epoch in range(num_epochs):
        model.train()
        for input_batch,output_batch in train_loader:
            global_step+=1
            examples_seen+=input_batch.shape[0]

            loss=Cross_Entropy_Loss(model,input_batch,output_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if global_step%eval_freq==0:
                    train_loss,valid_loss=evaluate(model,train_loader,val_loader)
                    train_accuracy,_,_=Calculate_Accuracy(model,train_loader)
                    valid_accuracy,_,_=Calculate_Accuracy(model,val_loader)

                    print(f"Epoch {epoch+1}/{num_epochs}, Step {global_step}/{len(train_loader)*num_epochs}, Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}, Train Acc: {train_accuracy:.4f}, Valid Acc: {valid_accuracy:.4f}")

                    train_losses.append(train_loss)
                    valid_losses.append(valid_loss)
                    train_acc.append(train_accuracy)
                    valid_acc.append(valid_accuracy)

    return train_losses,valid_losses,train_acc,valid_acc

In [33]:
%%time
optimizer=th.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.1)
train_losses,valid_losses,train_acc,valid_acc=Training_Loop(model,train_loader,valid_loader,optimizer,num_epochs=1,eval_freq=10)

Epoch 1/1, Step 0/105, Train Loss: 1.5760, Valid Loss: 1.6619, Train Acc: 0.4250, Valid Acc: 0.4500
Epoch 1/1, Step 10/105, Train Loss: 0.9529, Valid Loss: 0.9056, Train Acc: 0.5500, Valid Acc: 0.5750
Epoch 1/1, Step 20/105, Train Loss: 0.6648, Valid Loss: 0.6616, Train Acc: 0.5750, Valid Acc: 0.4750
Epoch 1/1, Step 30/105, Train Loss: 0.7684, Valid Loss: 0.7825, Train Acc: 0.6250, Valid Acc: 0.5000
Epoch 1/1, Step 40/105, Train Loss: 0.6409, Valid Loss: 0.6306, Train Acc: 0.5000, Valid Acc: 0.5750
Epoch 1/1, Step 50/105, Train Loss: 0.6231, Valid Loss: 0.6242, Train Acc: 0.7750, Valid Acc: 0.8000
Epoch 1/1, Step 60/105, Train Loss: 0.5984, Valid Loss: 0.5940, Train Acc: 0.8250, Valid Acc: 0.8250
Epoch 1/1, Step 70/105, Train Loss: 0.5859, Valid Loss: 0.5789, Train Acc: 0.7250, Valid Acc: 0.6750
Epoch 1/1, Step 80/105, Train Loss: 0.5447, Valid Loss: 0.5377, Train Acc: 1.0000, Valid Acc: 0.8250
Epoch 1/1, Step 90/105, Train Loss: 0.5190, Valid Loss: 0.5111, Train Acc: 0.8500, Valid Acc

In [46]:
input=next(iter(test_loader))[0][0].unsqueeze(0)

In [47]:
with th.no_grad():
    logits=model(input)[:,-1,:]
th.argmax(logits)

tensor(1)

In [43]:
input_ids = tokenizer.encode(input)

print(input_ids)
decoded_ids=generate_text(model,next(iter(test_loader))[0][0].unsqueeze(0),context_size=100,new_max_tokens=16)

[198, 220, 220, 220, 36509, 1708, 2420, 705, 2777, 321, 6, 393, 705, 2763, 6, 1058, 198, 220, 220, 220, 8255, 1058, 705, 1639, 389, 257, 8464, 345, 423, 587, 20905, 6163, 284, 3328, 720, 12825, 5003, 393, 257, 720, 11024, 5764, 532, 383, 8255, 318, 10090, 355, 220, 220, 220]


In [51]:
tokenizer.decode(input.squeeze(0).tolist())

'83039 62735=£450 UK Break AccommodationVouchers terms & conditions apply. 2 claim you mustprovide your claim number which is 15541 <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>'